# Gemma 4 on Colab, exposed through ngrok

Hosts Gemma 4 with Ollama and publishes it through an ngrok URL for the Node.js backend.

**Before running**
1. Runtime > Change runtime type > **T4 GPU**.
2. The ngrok authtoken is set directly in cell 6.
3. Run all cells in order. The public URL is printed by the ngrok cell.

Keep this tab open while using the backend. When the runtime stops, the URL stops working.

In [ ]:
# 1. Check the GPU
import subprocess
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True, check=True).stdout)
except Exception:
    print("No GPU found. Use Runtime > Change runtime type > T4 GPU, then run again.")

In [ ]:
# 2. Config
MODEL = "gemma4:e4b"   # safe default for a T4. "gemma4:12b" is the larger option.
NUM_CTX = 8192         # Ollama's default context is small, so it is set on every request

In [ ]:
# 3. Install Ollama and pyngrok
import subprocess

def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

sh("apt-get update -qq && apt-get install -y -qq zstd pciutils lshw")
sh("curl -fsSL https://ollama.com/install.sh | sh")
sh("pip install -q pyngrok requests")

In [ ]:
# 4. Start the Ollama server in the background
import os, time, subprocess, requests

server_env = {**os.environ, "OLLAMA_HOST": "0.0.0.0:11434", "OLLAMA_KEEP_ALIVE": "24h"}
client_env = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"}

server = subprocess.Popen(["ollama", "serve"], env=server_env,
                          stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start. Check ollama.log")

print("Ollama is running")

In [ ]:
# 5. Download the model (several GB, takes a few minutes the first time)
r = subprocess.run(["ollama", "pull", MODEL], env=client_env, text=True)
assert r.returncode == 0, "Model pull failed. Check the tag in the config cell."

In [ ]:
# 6. Open the ngrok tunnel
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = "3K7ynlSkmM2R6bd06baYQSBDKFC_6mNuKJowxY1NRdGR3rzfW"

# Pass the token through config instead of set_auth_token(), which errors if a token is already saved.
conf.get_default().auth_token = NGROK_AUTHTOKEN

# Close any tunnels left over from a previous run of this cell
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

tunnel = ngrok.connect(11434, "http")
PUBLIC_URL = tunnel.public_url
print("PUBLIC URL:", PUBLIC_URL)

In [ ]:
# 7. Smoke test through the public URL (same call the Node backend will make)
import json, time, requests

schema = {
    "type": "object",
    "properties": {
        "correctness": {"type": "integer"},
        "completeness": {"type": "integer"},
        "fluency": {"type": "integer"},
        "feedback": {"type": "string"},
        "missing_concepts": {"type": "array", "items": {"type": "string"}},
        "ideal_answer": {"type": "string"},
        "next_question": {"type": "string"},
    },
    "required": ["correctness", "completeness", "fluency", "feedback",
                 "missing_concepts", "ideal_answer", "next_question"],
}

system = (
    "You are a technical interviewer evaluating a spoken answer that was transcribed to text. "
    "Score correctness, completeness and fluency from 0 to 10. "
    "Fluency means clarity of thought: is the answer organised, coherent and easy to follow. "
    "Ignore filler words, restarts, grammar and accent. Do not let fluency affect correctness. "
    "Then ask one spoken-style follow-up question. Reply with JSON only."
)
user = (
    "Topic: JavaScript closures\n"
    "Question: What is a closure and when would you use one?\n"
    "Answer transcript: um so a closure is basically when a function remembers the variables "
    "from where it was created, uh, even after that outer function has finished. "
    "You use it for like private variables, and I think it copies the values when it is created."
)

payload = {
    "model": MODEL,
    "stream": False,
    "format": schema,
    "options": {"temperature": 0.1, "num_ctx": NUM_CTX},
    "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
}

start = time.time()
resp = requests.post(f"{PUBLIC_URL}/api/chat", json=payload,
                     headers={"ngrok-skip-browser-warning": "true"}, timeout=300)
resp.raise_for_status()
print(f"Took {time.time() - start:.1f}s")
print(json.dumps(json.loads(resp.json()["message"]["content"]), indent=2))

In [ ]:
# 8. Keep the runtime alive while you work (stop this cell when you are done)
import time, requests
print("Serving at:", PUBLIC_URL)
while True:
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=5)
    except Exception as e:
        print("Ollama check failed:", e)
    time.sleep(60)